# Step 3 · Fine-tuned text model (runs on Colab's free GPU)

Teaches a small language model (`BAAI/bge-small-en-v1.5`) to read a post and estimate **how much it beats its
author's usual level** (`y - lb`, log scale), for views and the X score.

**Fair by construction.** The folds come from the pipeline (`fold1..fold4` = train / cal / test). Each fold's model
trains **only** on that fold's `train` posts and predicts its `cal` and `test` posts. Back in the pipeline, these
predictions become one more feature, judged by the same backtest and keep/drop rule as everything else
(`reports/3_protocol.md`). If it doesn't significantly help, it's dropped.

**Steps**
1. Runtime → Change runtime type → **T4 GPU**.
2. On your PC: `uv run seeding export-training` → upload `data/colab/train_posts.parquet` below.
3. Run all cells (about 20–40 minutes).
4. Download `text_ft_preds.parquet`, put it in `data/colab/`, then on your PC:
   `uv run seeding forecast-backtest --members A_usual,C_1D,L1_lgbm,L1_lgbm_textft`

In [ ]:
!pip -q install "transformers>=4.44" "datasets" "accelerate" pyarrow
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4 GPU")

In [ ]:
from google.colab import files
up = files.upload()            # choose data/colab/train_posts.parquet
import pandas as pd, numpy as np, io
df = pd.read_parquet(io.BytesIO(next(iter(up.values()))))
print(len(df), "posts;", df.author.nunique(), "accounts")
print({f: df[f].value_counts().to_dict() for f in ["fold1", "fold2", "fold3", "fold4"]})

In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding)

MODEL = "BAAI/bge-small-en-v1.5"
TARGETS = ["views", "xscore"]
tok = AutoTokenizer.from_pretrained(MODEL)

def ds(frame, label=None):
    d = {"text": frame.text.fillna("").str.slice(0, 1000).tolist()}
    if label is not None:
        d["labels"] = label.astype("float32").tolist()
    return Dataset.from_dict(d).map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True,
                                    remove_columns=["text"])

out = []
for k in range(1, 5):
    f = f"fold{k}"
    tr, ev_ = df[df[f] == "train"], df[df[f].isin(["cal", "test"])]
    for t in TARGETS:
        trt = tr.dropna(subset=[f"y_{t}", f"lb_{t}"])
        resid = (trt[f"y_{t}"] - trt[f"lb_{t}"]).to_numpy()
        mu, sd = resid.mean(), resid.std()
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1, problem_type="regression")
        args = TrainingArguments(output_dir="/tmp/ft", per_device_train_batch_size=64, num_train_epochs=2,
                                 learning_rate=3e-5, warmup_ratio=0.06, weight_decay=0.01, fp16=True,
                                 logging_steps=200, save_strategy="no", report_to=[], seed=0)
        Trainer(model=model, args=args, train_dataset=ds(trt, (resid - mu) / sd),
                data_collator=DataCollatorWithPadding(tok)).train()
        pred = Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tok)).predict(ds(ev_))
        out.append(pd.DataFrame({"fold": k, "tweet_id": ev_.tweet_id.to_numpy(), "target": t,
                                 "split": ev_[f].to_numpy(), "ft_pred": pred.predictions.ravel() * sd + mu}))
        print(f"fold {k} {t}: trained on {len(trt):,}, predicted {len(ev_):,}")
        del model; torch.cuda.empty_cache()
preds = pd.concat(out, ignore_index=True)

In [ ]:
# quick sanity check on the cal posts (the real judgement happens in the pipeline's backtest)
from scipy.stats import spearmanr
for (k, t), g in preds[preds.split == "cal"].groupby(["fold", "target"]):
    m = g.merge(df[["tweet_id", f"y_{t}", f"lb_{t}"]], on="tweet_id")
    print(k, t, "Spearman with (y - usual):", round(spearmanr(m.ft_pred, m[f"y_{t}"] - m[f"lb_{t}"]).statistic, 3))
preds.to_parquet("text_ft_preds.parquet", index=False)
files.download("text_ft_preds.parquet")